In [1]:
from query_orders import get_spark_session

In [2]:
spark = get_spark_session("alice", "test1234", "lakehouse-local")
spark.sql("SHOW NAMESPACES").show()

26/08/06 00:38:28 WARN Utils: Your hostname, vincent-k-gpu resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/06 00:38:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/vkieuvongngam/.ivy2/cache
The jars for the packages stored in: /home/vkieuvongngam/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-696676d8-3b36-4866-b49d-e577059e3239;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/vkieuvongngam/exploration/spark-lakehouse/spark/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.0 in central
:: resolution report :: resolve 76ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-aws-bundle;1.10.0 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-696676d8-3b36-4866-b49d-e577059e3239
	confs: [default]
	0 artifacts copied, 2 already retrieved (0kB/4ms)
26/08/06 00:38:28 WARN NativeCodeLoader: Un

+---------+
|namespace|
+---------+
|    sales|
+---------+



In [3]:
df = spark.table("sales.orders")

In [4]:
df.limit(2).show()

+--------+----------+------+
|order_id|  customer|amount|
+--------+----------+------+
|       1| acme-corp|1250.0|
|       2|globex-inc| 430.5|
+--------+----------+------+



In [5]:
df.schema

StructType([StructField('order_id', LongType(), True), StructField('customer', StringType(), True), StructField('amount', DoubleType(), True)])

In [6]:
from datetime import datetime

In [7]:
def add_one(iterator):
    for pdf in iterator:
        pdf["double_amount"] = pdf["amount"] * 2
        pdf['order_ts'] = datetime.now()
        yield pdf[["order_id","customer","amount", "double_amount", "order_ts"]]

new_df = df.mapInPandas(add_one, schema="order_id: long, customer: string, amount: double, double_amount: double, order_ts: timestamp")

## append to a table already defined elsewhere

In [8]:
new_df.limit(2).show()

+--------+----------+------+-------------+--------------------+
|order_id|  customer|amount|double_amount|            order_ts|
+--------+----------+------+-------------+--------------------+
|       1| acme-corp|1250.0|       2500.0|2026-08-06 00:38:...|
|       2|globex-inc| 430.5|        861.0|2026-08-06 00:38:...|
+--------+----------+------+-------------+--------------------+



In [9]:
new_df.select("order_id","customer","double_amount","order_ts"
).withColumnRenamed("double_amount", "amount").write.format("iceberg"
).mode("append"
).save(
    "lakekeeper.sales.orders_v2"
)

## create a new table on the fly (if the create ability is granted)

In [8]:

new_df.writeTo("lakekeeper.sales.orders_v4").create()